In [356]:
import pandas as pd
import pathlib as pl
import json
import re

In [357]:
SOURCE_FILE_PATH_CPO = pl.Path("../../dataassets/IRR_data_raw/VW/VW Feb26 CPO(2025 CPO).csv")
SOURCE_FILE_PATH_CPO = SOURCE_FILE_PATH_CPO.resolve()
    
SOURCE_FILE_PATH = pl.Path("../../dataassets/IRR_data_raw/VW/VW APR26.xlsx")
SOURCE_FILE_PATH = SOURCE_FILE_PATH.resolve()

DB_DIR = pl.Path("../../database/dbs")
DB_DIR.mkdir(parents=True, exist_ok=True)

VW_MODEL_DB_PATH = DB_DIR / "vw_model_db.csv"




In [358]:
# Transformation rules
# Example config for changing "ID." to "IDE" in "model" column
config = [
    {
        "column": "Description",
        "action": "change_text",
        "keyword": "ID. Buzz",
        "replacement": "ID.Buzz"
    }
]



In [359]:
# Strip whitespace from column names and values in the combined dataframe
def strip_whitespace(df):
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()
        
    return df

In [360]:
# create a function that takes in a file path, loads either CSV or Excel, and returns a single dataframe.
# For Excel files, it also adds a 'worksheet' column so we can track source sheet names.
# Adds source_file and Year columns to the combined dataframe.

def extract_year_from_name(name):
    match = re.search(r"(\d{2})", str(name))
    if match and match.group(1).isdigit():
        short_year = int(match.group(1))
        if short_year<99:
            full_year = 2000 + short_year
            return full_year
        
        return "Year is greater than 99. Need to resolve this" 
    return "can't find a number"


def read_table_to_dataframe(file_path):
    file_path = pl.Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(file_path)
        df["worksheet"] = "csv"
        df["source_file"] = "csv"
        df["Year"] = extract_year_from_name("csv")
        return df

    if suffix in {".xls", ".xlsx", ".xlsm", ".xlsb", ".odf", ".ods", ".odt"}:
        sheets = pd.read_excel(file_path, sheet_name=None)
        df_list = []

        for sheet_name, sheet_df in sheets.items():
            sheet_df = sheet_df.copy()
            sheet_df["worksheet"] = sheet_name
            df_list.append(sheet_df)

        combined_df = pd.concat(df_list, ignore_index=True)
        combined_df["source_file"] = combined_df["worksheet"]
        combined_df["Year"] = combined_df["worksheet"].apply(extract_year_from_name)
        combined_df = strip_whitespace(combined_df)
        return combined_df

    raise ValueError(f"Unsupported file type: {file_path.suffix}. Please provide a .csv or Excel file.")


In [361]:
def update_vw_model_db(grouped_df, db_path="VW_model_db.csv"):
    """
    Creates VW_model_db.csv if it doesn't exist, then appends new unique models
    from the grouped DataFrame, ensuring no duplicates based on normalized model,
    year, and output Model values. Adds columns: Model, Description, Description2,
    source_file, Year, is_active_model.
    """

    required_columns = {"model", "Description", "Description2", "source_file", "Year"}
    missing = required_columns - set(grouped_df.columns)
    if missing:
        raise ValueError(f"grouped_df is missing required columns: {sorted(missing)}")

    db_path = pl.Path(db_path)

    if db_path.exists():
        existing_df = pd.read_csv(db_path)
        if {"Model", "Year"}.issubset(existing_df.columns):
            existing_keys = set(
                zip(
                    existing_df["Model"].astype(str).str.lower(),
                    existing_df["Year"].astype(str),
                    existing_df["Model"].astype(str)
                )
            )
        else:
            existing_keys = set()
    else:
        existing_keys = set()
        pd.DataFrame(columns=['Model', 'Description', 'Description2', 'source_file', 'Year', 'is_active_model']).to_csv(db_path, index=False)

    model_agg = (
        grouped_df
        .dropna(subset=['model'])
        .groupby(['model', 'Year', 'source_file'], as_index=False)
        .agg({
            'Description': 'first',
            'Description2': 'first'
        })
    )

    model_agg['model_key'] = model_agg['model'].astype(str).str.lower()
    model_agg['year_key'] = model_agg['Year'].astype(str)
    model_agg['output_model'] = model_agg['model_key'].str.upper()

    new_keys = list(zip(model_agg['model_key'], model_agg['year_key'], model_agg['output_model']))
    truly_new = model_agg.loc[~pd.Series(new_keys).isin(existing_keys)].copy()

    if not truly_new.empty:
        truly_new['Model'] = truly_new['output_model']
        truly_new['is_active_model'] = True
        new_records = truly_new[['Model', 'Description', 'Description2', 'source_file', 'Year', 'is_active_model']]
        new_records.to_csv(db_path, mode='a', header=False, index=False)
        print(f"Added {len(new_records)} new models to {db_path}")
    else:
        print(f"No new models to add to {db_path}")


In [362]:
# transform this function to take in model_name, year, and model_db_df as paramerters.

# fethc in db_df to find value that matches year and modle name in the 

In [363]:


def apply_config_transformations(df, config):
    """
    Applies data transformations based on a config structure.
    Config is a list of dicts, each with:
    - 'column': the column name to transform
    - 'action': the action to perform (e.g., 'change_text')
    - 'keyword': the text to look for
    - 'replacement': the text to replace with
    """
    df = df.copy()
    for rule in config:
        column = rule['column']
        action = rule['action']
        if action == 'change_text':
            keyword = rule['keyword']
            replacement = rule['replacement']
            if column in df.columns:
                df[column] = df[column].str.replace(keyword, replacement, regex=False)
                display(df[column].unique())  # Display unique values to verify transformation
    return df

In [364]:
combined_df = read_table_to_dataframe(SOURCE_FILE_PATH)



In [365]:
combined_df.columns

Index(['Model', 'Description', 'Description2', 'Package', 'LI', 'SL12', 'RV12',
       'SL24', 'RV24', 'SL30', 'RV30', 'SL36', 'RV36', 'SL42', 'RV42', 'SL48',
       'RV48', 'SL54', 'RV54', 'SL60', 'RV60', 'SLR12-60', 'EPL12-48', 'LLR',
       'AFAR', 'worksheet', 'SF12-24', 'SF25-36', 'SF37-48', 'SF49-60',
       'SF61-72', 'SF73-84', '1SR24-84', '1SR85-96', '2SR24-84', '2SR85-96',
       '3SR24-84', '3SR85-96', '4SR24-84', '4SR85-96', '5SR24-84', '5SR85-96',
       '6SR24-84', '6SR85-96', '7SR24-84', '7SR85-96', 'CC', 'FI', 'CI', 'DC',
       'MA', 'SLP', 'DED', 'AFAPM', 'VGRP', 'TYPM', 'DPFND', '2EPF12-84',
       '3EPF12-84', '5EPF12-84', '5EPF12-85', 'BF24', 'BF36', 'BF48', 'BF60',
       'SBF24-60', '2EPS24', '2EPS36', '2EPS48', '2EPS60', '3EPS24', '3EPS36',
       '3EPS48', '3EPS60', '5EPS24', '5EPS36', '5EPS48', '5EPS60', 'Column1',
       'SL51', 'RV51', 'EPL36', 'EPL48', 'EPL362', 'EPL483', 'VP',
       'source_file', 'Year'],
      dtype='object')

In [366]:
# Transformations

vw_models_df = strip_whitespace(combined_df)
vw_models_df["Description"] = vw_models_df["Description"].str.replace("ID. Buzz", "ID.Buzz", regex=False)  # adding this to standardize the model name across sources.
vw_models_df["Description2"] = vw_models_df["Description2"].str.replace("ID. Buzz", "ID.Buzz", regex=False)  # adding this to standardize the model name across sources.

grouped_model_trim_combo_count = (
    vw_models_df
    .groupby(["Description", "Description2", "source_file", "Year"], dropna=False)
    .agg(
        count=("Description", "size"),
        model=("Model", "first")
    )
    .reset_index()
)


In [367]:
update_vw_model_db(grouped_model_trim_combo_count, VW_MODEL_DB_PATH)

No new models to add to ..\..\database\dbs\vw_model_db.csv


In [368]:
grouped_model_trim_combo_count[grouped_model_trim_combo_count["Description"].str.contains("Tiguan", case=False, na=False)]



,Description,Description2,source_file,Year,count,model
138,Tiguan,Comfortline,MAR 25 FINANCE,2025,1,RM13PJ
139,Tiguan,Comfortline,MAR 25 LEASE,2025,1,RM13PJ
140,Tiguan,Comfortline,MAR 25 SELECT,2025,1,RM13PJ
141,Tiguan,Comfortline,MAR 26 FINANCE,2026,1,RM13PJ
142,Tiguan,Comfortline,MAR 26 LEASE,2026,1,RM13PJ
143,Tiguan,Comfortline,MAR 26 SELECT,2026,1,RM13PJ
144,Tiguan,Comfortline R-Line Black,MAR 25 FINANCE,2025,1,RM1VPJ
145,Tiguan,Comfortline R-Line Black,MAR 25 LEASE,2025,1,RM1VPJ
146,Tiguan,Comfortline R-Line Black,MAR 25 SELECT,2025,1,RM1VPJ
147,Tiguan,Comfortline R-Line Black,MAR 26 FINANCE,2026,1,RM1VPJ


In [369]:
model_db_df = pd.read_csv(VW_MODEL_DB_PATH)
model_db_df.head()

,Model,Description,Description2,source_file,Year,is_active_model
0,BU52RS,Jetta,Trendline,MAR 25 FINANCE,2025,True
1,BU52RS,Jetta,Trendline,MAR 25 LEASE,2025,True
2,BU52RS,Jetta,Trendline,MAR 25 SELECT,2025,True
3,BU52RS,Jetta,Trendline,MAR 26 FINANCE,2026,True
4,BU52RS,Jetta,Trendline,MAR 26 LEASE,2026,True


In [370]:
model_db_df[model_db_df["Description"].str.contains("Buzz", case=False, na=False)]

,Model,Description,Description2,source_file,Year,is_active_model
132,EBJR5S,ID.Buzz,ID.Buzz 1st Edition 4MOTION,MAR 25 FINANCE,2025,True
133,EBJR5S,ID.Buzz,ID.Buzz 1st Edition 4MOTION,MAR 25 LEASE,2025,True
134,EBJR5S,ID.Buzz,ID.Buzz 1st Edition 4MOTION,MAR 25 SELECT,2025,True
135,EBJR7S,ID.Buzz,ID.Buzz 1st Edition,MAR 25 FINANCE,2025,True
136,EBJR7S,ID.Buzz,ID.Buzz 1st Edition,MAR 25 LEASE,2025,True
137,EBJR7S,ID.Buzz,ID.Buzz 1st Edition,MAR 25 SELECT,2025,True
